# $D^0 \to K^\mp \pi^\pm \pi^\pm \pi^\mp$: the LHCb models from AmpGen

The amplitude models of LHCb, Eur. Phys. J. C 78 (2018) 443, for the Cabibbo-favoured
(CF) $D^0 \to K^-\pi^+\pi^+\pi^-$ and the doubly Cabibbo-suppressed (DCS)
$D^0 \to K^+\pi^-\pi^-\pi^+$ decays, as defined in
[AmpGen](https://github.com/GooFit/AmpGen)'s `options/D02Kpipipi.opt` and
`D02piKpipi.opt` and built as SymPy expressions by `aman_examples.ampgen`. The two
identical pions are symmetrised as in AmpGen. Each of the 33 CF and 15 DCS decay chains
agrees with AmpGen to about $10^{-5}$ of its rms (`tests/test_ampgen_d02k3pi.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.function.sympy import create_parametrized_function

from aman_examples import d02k3pi

## Models

In [ ]:
models = {"CF": d02k3pi.formulate_cf_model(), "DCS": d02k3pi.formulate_dcs_model()}
for key, model in models.items():
    print(f"{key}: {len(model.chains)} chains, final state {model.final_state}")

## Fit fractions and projections

The fit fraction of each $D^0$ decay, $\int |A_k|^2 / \int |A|^2$, on uniform phase
space, where $A_k$ is the sum of the chains with the coupling $c_k$ of that $D^0$ rule
(AmpGen expands, for example, $a_1(1260)^+ \to \rho\pi, (\pi\pi)_S\pi$ into several
chains).

In [ ]:
rng = NumpyUniformRNG(seed=1)
samples, intensities = {}, {}
for key, model in models.items():
    phsp = d02k3pi.phase_space_generator(model).generate(100_000, rng)
    groups = {}
    for name, expr in model.chains.items():
        coupling = model.couplings[name]
        rule = next(s for s in coupling.free_symbols if s.name.startswith("C_{D0"))
        groups[rule] = groups.get(rule, 0) + coupling * expr
    terms = {}
    for rule, expr in groups.items():
        function = create_parametrized_function(
            expr.doit(), model.parameter_defaults, backend="jax"
        )
        terms[rule.name[3:-1]] = np.asarray(function(phsp))
    total = sum(terms.values())
    norm = np.mean(np.abs(total) ** 2)
    fractions = {n: 100 * np.mean(np.abs(t) ** 2) / norm for n, t in terms.items()}
    print(f"{key}: sum of fit fractions {sum(fractions.values()):.1f}%")
    for name, value in sorted(fractions.items(), key=lambda x: -x[1]):
        print(f"  {value:5.1f}%  {name}")
    samples[key], intensities[key] = phsp, np.abs(total) ** 2

In [ ]:
def mass(sample, indices):
    p = sum(np.column_stack([sample[f"p{i}_{c}"] for c in "xyzE"]) for i in indices)
    return np.sqrt(p[:, 3] ** 2 - np.sum(p[:, :3] ** 2, axis=1))


projections = [
    (r"$m(K\pi)$ [opposite-sign]", (0, 1)),
    (r"$m(\pi\pi)$ [opposite-sign]", (1, 3)),
    (r"$m(K\pi\pi)$", (0, 1, 3)),
    (r"$m(\pi\pi\pi)$", (1, 2, 3)),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), tight_layout=True)
for ax, (label, indices) in zip(axes, projections, strict=True):
    for key, sample in samples.items():
        w = intensities[key]
        ax.hist(
            mass(sample, indices),
            bins=80,
            weights=w / w.sum(),
            histtype="step",
            label=key,
        )
    ax.set_xlabel(f"{label} [GeV]")
axes[0].legend()
plt.show()